In [23]:
# ==========================================
# 1. CONFIGURATION & GLOBAL DATA LOAD
# ==========================================
import os
import re
import json
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# File paths and naming
EXPERIMENT = "test_8_simulation2"
DATA_FOLDER = "TH0011AV"           # Folder containing the point clouds and listed in the CSV
CAD_MODEL_DIR = "TH0011AV"      # Folder containing the .stl feature models (e.g. workpiece3, workpiece31)

ENABLE_VISUALIZATION = True

USE_GROUND_TRUTH = True
GT_CSV_PATH = f"processed_data/{EXPERIMENT}/metadata.csv"

CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_moe.csv"

WORKPIECE_PATH = f"workpiece/{CAD_MODEL_DIR}/workpiece.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}"

# The EXACT 40k points saved from 1_3_viewpoint_generation_manual
GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}/pcd_all.pcd"
GLOBAL_COVERED_JSON = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}/covered_indices.json"

# Optimization Settings
OPTIMIZATION_METHOD = "GRASP"  # "GREEDY" or "GRASP"
GRASP_ITERATIONS = 20          # Number of sequences to generate
RCL_SIZE = 3                   # Top N candidates to pick randomly from (Cardinality-based RCL)

# Utility weights & Discretization
ALPHA = 0.9 # Coverability
BETA = 0.1 # Confidence
GAMMA = 0.5  # Submodular decay factor for Coverability
TOTAL_STEPS = 12  # Number of viewpoints to select
# DISTANCE_THRESHOLD = 2.0  # mm
AXIS_SIZE = 20.0
NUM_BINS = 10
TOP_PERCENTILE_FILTER = 0.2 # Keep the top n% of viewpoints based on Score

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            poses[idx] = np.load(os.path.join(folder_path, f))
    return poses

# ==========================================
# 3. LOAD GLOBAL DATA & MULTIPLE FEATURES
# ==========================================
print("Loading Global 40k Point Cloud...")
pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH) if os.path.exists(GLOBAL_PCD_PATH) else o3d.geometry.PointCloud()
pcd_all.paint_uniform_color([0.6, 0.6, 0.6])

print("Loading Global Viewpoint Visibilities...")
if os.path.exists(GLOBAL_COVERED_JSON):
    with open(GLOBAL_COVERED_JSON, "r") as f:
        global_visibility_dict = json.load(f)
else:
    global_visibility_dict = {}

poses_dict = load_viewpoint_poses_dict(POSES_PATH)

# Load Inference Results CSV and parse Filename column
if USE_GROUND_TRUTH:
    df_inference = pd.read_csv(GT_CSV_PATH)
    df_inference.rename(columns={'filename': 'Filename', 'chamfer_value': 'Predicted_CD'}, inplace=True)
    print(f"\n--- USING GROUND TRUTH DATA FROM {GT_CSV_PATH} ---")
else:
    df_inference = pd.read_csv(CSV_PATH)
    print(f"\n--- USING PREDICTED DATA FROM {CSV_PATH} ---")
df_inference[['Parsed_Folder', 'Viewpoint', 'Feature']] = df_inference['Filename'].str.extract(r'(.*?)/viewpoint_simulated_(\d+)_(.*?)\.pcd')
df_inference['Viewpoint'] = df_inference['Viewpoint'].astype(int)

# Filter for the requested workpiece folder
df_chamfer = df_inference[df_inference['Parsed_Folder'] == DATA_FOLDER].copy()
df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)

# --- NEW: Calculate Global Normalized Feature Confidence ---
min_cd = df_chamfer[df_chamfer['Chamfer_Distance_mm'] >= 0]['Chamfer_Distance_mm'].min()
max_cd = df_chamfer['Chamfer_Distance_mm'].max()
if max_cd > min_cd:
    df_chamfer['Feature_Confidence'] = 1.0 - ((df_chamfer['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
else:
    df_chamfer['Feature_Confidence'] = 1.0

features = df_chamfer['Feature'].unique()
print(f"\nFound {len(features)} unique features to optimize for {DATA_FOLDER}: {features}")

feature_indices = {}
all_features_indices = set()

# 1. Load all features simultaneously
feature_pcds = []
feature_names = []
for feat in features:
    feat_path = f"workpiece/{CAD_MODEL_DIR}/{feat}.stl"
    if os.path.exists(feat_path):
        f_mesh = o3d.io.read_triangle_mesh(feat_path)
        f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
        feature_pcds.append(f_pcd)
        feature_names.append(feat)
    else:
        print(f"  - WARNING: {feat_path} not found!")

# 2. Voronoi Nearest-Neighbor Competition
if len(feature_pcds) > 0 and len(pcd_all.points) > 0:
    dists = np.zeros((len(pcd_all.points), len(feature_pcds)))
    for i, f_pcd in enumerate(feature_pcds):
        dists[:, i] = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
    
    # Assign each point to the closest feature
    assignments = np.argmin(dists, axis=1)
    
    # 3. Populate feature indices
    for i, feat in enumerate(feature_names):
        idx_set = set(np.where(assignments == i)[0])
        feature_indices[feat] = idx_set
        all_features_indices.update(idx_set)
        print(f"  - {feat}: {len(idx_set)} matching points in 40k cloud (Voronoi Assignment)")
else:
    print("WARNING: No valid features or empty global point cloud!")

print(f"\nTotal target points across all features: {len(all_features_indices)}")


Loading Global 40k Point Cloud...
Loading Global Viewpoint Visibilities...

--- USING GROUND TRUTH DATA FROM processed_data/test_8_simulation2/metadata.csv ---

Found 3 unique features to optimize for TH0011AV: ['surface0' 'surface1' 'surface2']
  - surface0: 4984 matching points in 40k cloud (Voronoi Assignment)
  - surface1: 8325 matching points in 40k cloud (Voronoi Assignment)
  - surface2: 26691 matching points in 40k cloud (Voronoi Assignment)

Total target points across all features: 40000


In [24]:
# ==========================================
# 4. OPTIMIZATION RUN (GREEDY / GRASP)
# ==========================================
import pandas as pd
import numpy as np
import random

if OPTIMIZATION_METHOD == "GREEDY":
    num_iterations = 1
    rcl_size = 1
else:
    num_iterations = GRASP_ITERATIONS
    rcl_size = RCL_SIZE

if len(all_features_indices) == 0:
    print("WARNING: all_features_indices is empty. Make sure you loaded the data correctly.")

print(f"Starting {OPTIMIZATION_METHOD} Optimization for {TOTAL_STEPS} steps...")
if OPTIMIZATION_METHOD == "GRASP":
    print(f"Running {num_iterations} iterations with RCL Size = {rcl_size} (Cardinality)...\n")

# Global tracking for the absolute best sequence found
best_overall_sequence = []
best_overall_score = -1.0
best_overall_point_counts = None
best_step_to_covered = {}
best_all_step_results = []

# --- PRECOMPUTE STATIC CHAMFER DATA ---
print("Precomputing static viewpoint data (Dynamic Filter)...")
total_object_points = len(all_features_indices)
static_viewpoint_data = {}
temp_cameras = []

for v_idx, group in df_chamfer.groupby('Viewpoint'):
    v_idx = int(v_idx)
    if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]): continue
    
    camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
    
    camera_target_indices = set()
    sum_weighted_chamfer = 0.0
    sum_points = 0
    
    for _, row in group.iterrows():
        feature = row['Feature']
        chamfer_dist = row['Chamfer_Distance_mm']
        if feature not in feature_indices: continue
            
        feat_visible_points = camera_visible_40k & feature_indices[feature]
        P_vj = len(feat_visible_points)
        if P_vj == 0: continue
            
        camera_target_indices.update(feat_visible_points)
        sum_weighted_chamfer += (P_vj * chamfer_dist)
        sum_points += P_vj
        
    if sum_points == 0: continue
        
    weighted_chamfer = sum_weighted_chamfer / sum_points
    
    temp_cameras.append({
        'v_idx': v_idx,
        'weighted_chamfer': weighted_chamfer,
        'visible_target_indices': camera_target_indices,
        'sum_points': sum_points
    })

# Normalize and Score
if len(temp_cameras) > 0:
    min_cam_cd = min(c['weighted_chamfer'] for c in temp_cameras)
    max_cam_cd = max(c['weighted_chamfer'] for c in temp_cameras)
    
    for c in temp_cameras:
        if max_cam_cd > min_cam_cd:
            c['confidence'] = 1.0 - ((c['weighted_chamfer'] - min_cam_cd) / (max_cam_cd - min_cam_cd))
        else:
            c['confidence'] = 1.0
            
        c['filter_score'] = (c['sum_points'] / total_object_points) * c['confidence']
        
    # Sort by Filter Score (Highest to Lowest)
    temp_cameras.sort(key=lambda x: x['filter_score'], reverse=True)
    num_to_keep = max(1, int(len(temp_cameras) * TOP_PERCENTILE_FILTER))
    top_cameras = temp_cameras[:num_to_keep]
    
    for c in top_cameras:
        static_viewpoint_data[c['v_idx']] = {
            'weighted_chamfer': c['weighted_chamfer'],
            'visible_target_indices': c['visible_target_indices'],
            'idx_arr': list(c['visible_target_indices'])
        }
    print(f"Filtered to Top {TOP_PERCENTILE_FILTER*100}% cameras ({len(static_viewpoint_data)}/{len(temp_cameras)} cameras).")
else:
    print("No valid cameras found!")

for iteration in range(1, num_iterations + 1):
    
    # State is completely reset at the start of every sequence generation
    point_coverage_counts = np.zeros(len(pcd_all.points))
    selected_viewpoints = []
    step_to_covered_indices = {}
    all_step_results = []
    
    cumulative_utility = 0.0
    
    for k in range(1, TOTAL_STEPS + 1):
        db_records = []
        
        for v_idx, data in static_viewpoint_data.items():
            if v_idx in selected_viewpoints:
                continue
                
            idx_arr = data['idx_arr']
            c_vals = point_coverage_counts[idx_arr]
            submodular_sum = np.sum(GAMMA ** c_vals)
            coverability = submodular_sum / len(all_features_indices)
            
            # Vectorized newly covered calculation
            newly_covered_arr = np.array(idx_arr)[c_vals == 0]
            newly_covered = newly_covered_arr.tolist()
            
            db_records.append({
                'Step': k,
                'Viewpoint': v_idx,
                'Coverability': coverability,
                'Chamfer_Distance': data['weighted_chamfer'],
                '_visible_target_indices': data['visible_target_indices'],
                '_new_points_set': newly_covered
            })
        
        df_step = pd.DataFrame(db_records)
        if df_step.empty:
            break
            
        # --- Normalize Metrics ---
        min_cov = df_step['Coverability'].min()
        max_cov = df_step['Coverability'].max()
        df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
        
        # --- Uncertainty & Confidence ---
        df_step['Uncertainty'] = df_step['Chamfer_Distance']
        min_uncert = df_step['Uncertainty'].min()
        max_uncert = df_step['Uncertainty'].max()
        df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
        
        # --- Utility Score ---
        df_step['Information_Gain'] = df_step['Norm_Coverability']
        df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
        
        # Rank Candidates
        df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
        df_step['Rank'] = df_step.index + 1
        
        # --------------------------------------------------
        # RCL SELECTION (CARDINALITY)
        # --------------------------------------------------
        actual_rcl_size = min(rcl_size, len(df_step))
        rcl = df_step.head(actual_rcl_size)
        
        # Pick completely randomly from the RCL
        chosen_idx = random.randint(0, actual_rcl_size - 1)
        best_row = rcl.iloc[chosen_idx]
        
        all_step_results.append(best_row)
        
        best_v_idx = int(best_row['Viewpoint'])
        selected_viewpoints.append(best_v_idx)
        
        # Update state
        idx_arr = np.array(list(best_row['_visible_target_indices']), dtype=int)
        point_coverage_counts[idx_arr] += 1
        cumulative_utility += best_row['Utility_Score']
        
        step_to_covered_indices[k] = list(best_row['_visible_target_indices'])
    
    if cumulative_utility > best_overall_score:
        best_overall_score = cumulative_utility
        best_overall_sequence = list(selected_viewpoints)
        best_overall_point_counts = point_coverage_counts.copy()
        best_step_to_covered = dict(step_to_covered_indices)
        best_all_step_results = list(all_step_results)
        
    if OPTIMIZATION_METHOD == "GRASP":
        print(f"  Iteration {iteration}/{num_iterations} | Current Utility: {cumulative_utility:.4f} | Best Sum Utility: {best_overall_score:.4f}")

# --- FINAL RESULTS ---
selected_viewpoints = best_overall_sequence
point_coverage_counts = best_overall_point_counts
step_to_covered_indices = best_step_to_covered
all_step_results = best_all_step_results

print(f"\n{'='*50}")
print(f"Processing Workpiece: {DATA_FOLDER}")
print(f"{'='*50}")
print(f"Manual Viewpoint Sequence: {selected_viewpoints}\n")

table_data = []
for result in all_step_results:
    table_data.append({
        'Step': int(result['Step']),
        'Viewpoint': int(result['Viewpoint']),
        'Utility': result['Utility_Score'],
        'Cumulative Coverability': result['Coverability'],
        'Raw Error (mm)': result['Chamfer_Distance'],
        'Normalized Confidence': result['Confidence']
    })

df_table = pd.DataFrame(table_data)
df_table.set_index('Step', inplace=True)
from IPython.display import display
display(df_table)


Starting GRASP Optimization for 12 steps...
Running 20 iterations with RCL Size = 3 (Cardinality)...

Precomputing static viewpoint data (Dynamic Filter)...
Filtered to Top 20.0% cameras (86/432 cameras).
  Iteration 1/20 | Current Utility: 10.8173 | Best Sum Utility: 10.8173
  Iteration 2/20 | Current Utility: 10.8209 | Best Sum Utility: 10.8209
  Iteration 3/20 | Current Utility: 10.8834 | Best Sum Utility: 10.8834
  Iteration 4/20 | Current Utility: 10.8691 | Best Sum Utility: 10.8834
  Iteration 5/20 | Current Utility: 10.9608 | Best Sum Utility: 10.9608
  Iteration 6/20 | Current Utility: 10.8776 | Best Sum Utility: 10.9608
  Iteration 7/20 | Current Utility: 10.8872 | Best Sum Utility: 10.9608
  Iteration 8/20 | Current Utility: 10.8467 | Best Sum Utility: 10.9608
  Iteration 9/20 | Current Utility: 10.9040 | Best Sum Utility: 10.9608
  Iteration 10/20 | Current Utility: 10.8367 | Best Sum Utility: 10.9608
  Iteration 11/20 | Current Utility: 10.8785 | Best Sum Utility: 10.9608
 

,Viewpoint,Utility,Cumulative Coverability,Raw Error (mm),Normalized Confidence
Step,,,,,
1,18,0.901320,0.530300,6.270484,0.013203
2,89,0.918365,0.386300,5.552115,0.524620
3,34,0.906022,0.206237,6.163628,0.089275
4,74,0.915720,0.164866,5.730138,0.397883
5,63,0.917716,0.087722,6.001156,0.204942
6,135,0.912329,0.077588,6.115853,0.123287
7,126,0.916860,0.041280,5.950016,0.241349
8,69,0.894110,0.037942,6.150525,0.098603
9,137,0.958043,0.020815,5.473723,0.580429


In [25]:
# ==========================================
# 5. VISUALIZATION COVERAGE PER VIEWPOINT
 # ==========================================
import copy
import matplotlib.pyplot as plt

# Create a copy of pcd_all to colorize
vis_pcd = copy.deepcopy(pcd_all)

# Default color (light gray) for unseen points
colors = np.ones((len(vis_pcd.points), 3)) * 0.8

# Generate distinct colors for each step (e.g. from tab10 colormap)
cmap = plt.get_cmap("tab10")

print("Point Colors:")
for k in range(1, TOTAL_STEPS + 1):
    if k in step_to_covered_indices:
        step_color = cmap(k - 1)[:3]  # RGB from colormap
        
        # Color text for the print statement using ANSI escape codes
        r, g, b = [int(c * 255) for c in step_color]
        colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
        print(f"{colored_text} covered {len(step_to_covered_indices[k])} new points.")
        
        indices = list(step_to_covered_indices[k])
        colors[indices] = step_color

vis_pcd.colors = o3d.utility.Vector3dVector(colors)

# Draw geometries
print("\nOpening Open3D visualization window...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        [vis_pcd], 
        window_name="Step-by-Step Coverage",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Point Colors:
Step 1 covered 21157 new points.
Step 2 covered 10578 new points.
Step 3 covered 159 new points.
Step 4 covered 48 new points.

Opening Open3D visualization window...


In [44]:
# ==========================================
# 6. EXTRA: VISUALIZE ONLY DETECTED POINTS
# ==========================================
import open3d as o3d
import numpy as np

print("Filtering out unseen (grey) points...")

# Collect all indices of points covered in ANY step
all_covered_indices = set()
for step, indices in step_to_covered_indices.items():
    all_covered_indices.update(indices)

# Use Open3D's select_by_index to extract ONLY the colored points
covered_pcd = vis_pcd.select_by_index(list(all_covered_indices))

print(f"Total points shown: {len(covered_pcd.points)}")

# Draw geometries
print("\nOpening Open3D visualization window (Detected Points Only)...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        [covered_pcd], 
        window_name="Detected Points Only",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Filtering out unseen (grey) points...
Total points shown: 31903

Opening Open3D visualization window (Detected Points Only)...


In [62]:
# ==========================================
# 7. VISUALIZE SELECTED GRASP POSES
# ==========================================
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import copy


vis_pcd = copy.deepcopy(pcd_all)

print("Generating visualization of the entire workpiece and camera poses...")

# We use the full workpiece (vis_pcd) instead of filtering it!
geometries = [vis_pcd]

cmap = plt.get_cmap("tab10")

print("\n--- Selected Viewpoints Summary ---")
# For each step, create the camera pose arrows matching the step color
for k, v_idx in enumerate(selected_viewpoints, 1):
    step_color = cmap(k - 1)[:3]
    
    # Print the summary
    r, g, b = [int(c * 255) for c in step_color]
    colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
    print(f"{colored_text}: Viewpoint {v_idx}")
    
    # Add the colored pose arrows to the visualization
    if v_idx in poses_dict:
        pose_arrows = make_xz_arrows(transform=poses_dict[v_idx], size=AXIS_SIZE, color=step_color)
        geometries.append(pose_arrows)

print("\nOpening Open3D visualization window...")

if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        geometries, 
        window_name="Entire Workpiece and Camera Poses",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Generating visualization of the entire workpiece and camera poses...

--- Selected Viewpoints Summary ---
Step 1: Viewpoint 34
Step 2: Viewpoint 89
Step 3: Viewpoint 47
Step 4: Viewpoint 182

Opening Open3D visualization window...


In [154]:
# ==========================================
# 8. STEP-BY-STEP OBSERVATION FREQUENCY
# ==========================================
import copy
import matplotlib.pyplot as plt
import open3d as o3d
import numpy as np

print("Generating progressive observation frequency visualizations...")
cmap = plt.get_cmap("tab10")

# We will simulate the frequency accumulation step-by-step
current_counts = np.zeros(len(pcd_all.points))

for k, v_idx in enumerate(selected_viewpoints, 1):
    print(f"\n{'='*40}")
    print(f" STEP {k} FREQUENCY STATE (After adding Viewpoint {v_idx})")
    print(f"{'='*40}")
    
    # 1. Fetch the exact points seen by this specific viewpoint
    camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
    visible_target_indices = camera_visible_40k & all_features_indices
    
    # 2. Add them to our cumulative counts
    for idx in visible_target_indices:
        current_counts[idx] += 1
        
    # 3. Create point cloud for this step
    freq_pcd = copy.deepcopy(pcd_all)
    colors = np.ones((len(freq_pcd.points), 3)) * 0.8
    
    max_obs = int(np.max(current_counts))
    for count in range(1, max_obs + 1):
        indices = np.where(current_counts == count)[0]
        if len(indices) > 0:
            freq_color = cmap(count - 1)[:3]
            r, g, b = [int(c * 255) for c in freq_color]
            colored_text = f"\033[38;2;{r};{g};{b}mSeen {count} Time(s)\033[0m"
            print(f"{colored_text}: {len(indices)} points")
            colors[indices] = freq_color
            
    freq_pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # 4. Filter out unseen points (0 count)
    observed_indices = np.where(current_counts > 0)[0]
    covered_freq_pcd = freq_pcd.select_by_index(list(observed_indices))
    
    print(f"Opening Open3D visualization window for Step {k}...")
    print("NOTE: Close the Open3D window to proceed to the next step!")
    
    o3d.visualization.draw_geometries(
        [covered_freq_pcd], 
        window_name=f"Observation Frequency (After Step {k})",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Generating progressive observation frequency visualizations...

 STEP 1 FREQUENCY STATE (After adding Viewpoint 36)
Seen 1 Time(s): 11809 points
Opening Open3D visualization window for Step 1...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 2 FREQUENCY STATE (After adding Viewpoint 169)
Seen 1 Time(s): 2990 points
Seen 2 Time(s): 9915 points
Opening Open3D visualization window for Step 2...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 3 FREQUENCY STATE (After adding Viewpoint 16)
Seen 1 Time(s): 1095 points
Seen 2 Time(s): 3026 points
Seen 3 Time(s): 8976 points
Opening Open3D visualization window for Step 3...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 4 FREQUENCY STATE (After adding Viewpoint 4)
Seen 1 Time(s): 159 points
Seen 2 Time(s): 1708 points
Seen 3 Time(s): 2433 points
Seen 4 Time(s): 8797 points
Opening Open3D visualization window for Step 4...
NOTE: Close the Open3D window to proceed to the next step!


In [14]:
# ==========================================
# 9. ENTIRE POSE AND CONFIDENCE VISUALIZATION
# ==========================================
import os
import re
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# 1. HELPER FUNCTIONS
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def visualize_viewpoints_colored(meshes, matrices, colors_per_frame, axis_size=50.0):
    if isinstance(matrices, np.ndarray) and matrices.ndim == 2:
        matrices = [matrices]

    geometries = list(meshes)
    geometries.append(make_xz_arrows(transform=None, size=axis_size, color=(0.9, 0.9, 0.9)))

    for i, (mat, color) in enumerate(zip(matrices, colors_per_frame)):
        geometries.append(make_xz_arrows(transform=mat, size=axis_size, color=color))

    print(f"\nVisualizing {len(matrices)} viewpoint(s)...")
    if ENABLE_VISUALIZATION:
        o3d.visualization.draw_geometries(
            geometries,
            window_name="Optimization Results & Viewpoint Visualization",
            width=1024, height=768,
            front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
        )

# 2. LOAD MESH / PCD
# We will just use `pcd_all` from the optimization block!
if 'pcd_all' not in locals() or len(pcd_all.points) == 0:
    print("Warning: pcd_all is empty. Run the optimization block first.")
    workpiece_pcd = o3d.geometry.PointCloud()
else:
    workpiece_pcd = pcd_all
    workpiece_pcd.paint_uniform_color([0.6, 0.6, 0.6])

# 3. POINT-WEIGHTED CONFIDENCE (FROM OPTIMIZER)
if 'static_viewpoint_data' not in locals() or len(static_viewpoint_data) == 0:
    print("Warning: static_viewpoint_data is empty. Please run the Optimization block first!")
    df_avg_conf = pd.DataFrame(columns=['Viewpoint', 'Feature_Confidence'])
else:
    records = []
    for v_idx, data in static_viewpoint_data.items():
        records.append({
            'Viewpoint': v_idx,
            'Chamfer_Distance_mm': data['weighted_chamfer']
        })
    df_avg_conf = pd.DataFrame(records)
    
    min_cd = df_avg_conf['Chamfer_Distance_mm'].min()
    max_cd = df_avg_conf['Chamfer_Distance_mm'].max()
    if max_cd > min_cd:
        df_avg_conf['Feature_Confidence'] = 1.0 - ((df_avg_conf['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
    else:
        df_avg_conf['Feature_Confidence'] = 1.0

# 4. COLOR PROCESSING (NUM_BINS)
if not df_avg_conf.empty:
    _cmap = plt.get_cmap('coolwarm')
    CLASS_COLORS = {cls: tuple(_cmap(cls / (NUM_BINS - 1))[:3]) for cls in range(NUM_BINS)}
    
    # Handle edge case where all confidences are identical
    if df_avg_conf['Feature_Confidence'].nunique() > 1:
        _, bins = pd.cut(df_avg_conf['Feature_Confidence'], bins=NUM_BINS, retbins=True)
        df_avg_conf['Confidence_Class'] = pd.cut(
            df_avg_conf['Feature_Confidence'], bins=bins, labels=range(NUM_BINS), include_lowest=True
        )
    else:
        df_avg_conf['Confidence_Class'] = NUM_BINS - 1
    
    valid_poses = []
    colors_per_frame = []
    
    for _, row in df_avg_conf.iterrows():
        v_idx = int(row['Viewpoint'])
        
        if v_idx in poses_dict and pd.notna(row['Confidence_Class']):
            # Skip drawing the cool/warm arrow if it's our optimal green arrow (prevents Z-fighting glitch)
            if 'selected_viewpoints' in locals() and v_idx in selected_viewpoints:
                continue
            valid_poses.append(poses_dict[v_idx])
            colors_per_frame.append(CLASS_COLORS[int(row['Confidence_Class'])])

    print(f"\nVisualizing {len(valid_poses)} valid viewpoints that passed the threshold filter.")
    print(f"Average Point-Weighted Confidence range: {df_avg_conf['Feature_Confidence'].min():.2f} - {df_avg_conf['Feature_Confidence'].max():.2f}")

    # --- HIGHLIGHT THE OPTIMAL SEQUENCE ---
    # Overwrite the colors of the chosen sequence to Bright Green so they stand out!
    # if 'selected_viewpoints' in locals() and selected_viewpoints:
    #     print(f"Highlighting Optimal Sequence in Bright Green: {selected_viewpoints}")
    #     for v_idx in selected_viewpoints:
    #         if v_idx in poses_dict:
    #             valid_poses.append(poses_dict[v_idx])
    #             colors_per_frame.append((0.0, 1.0, 0.0))

    # 5. SHOW RESULTS
    if valid_poses:
        visualize_viewpoints_colored([workpiece_pcd], valid_poses, colors_per_frame, axis_size=AXIS_SIZE)
    else:
        print("No valid viewpoint poses to visualize.")



Visualizing 82 valid viewpoints that passed the threshold filter.
Average Point-Weighted Confidence range: 0.00 - 1.00

Visualizing 82 viewpoint(s)...


### 10. Batch Processing (Multiple Workpieces)
Run the optimization pipeline automatically across an entire list of workpieces.

In [40]:
# ==========================================
# BATCH PROCESSING MULTIPLE WORKPIECES
# ==========================================
import os
import re
import json
import numpy as np
import pandas as pd
import open3d as o3d
import random
import time
    
# ------------------------------------------
# BATCH CONFIGURATION
# ------------------------------------------
BATCH_WORKPIECES = ["TH0032AV"]  # Add as many as you want to test
# BATCH_WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
EXPERIMENT = "test_8_simulation2"

USE_GROUND_TRUTH = True
# GT_CSV_PATH = f"processed_data/{EXPERIMENT}/metadata.csv"
CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_moe.csv"

OPTIMIZATION_METHOD = "GRASP"
GRASP_ITERATIONS = 50
RCL_SIZE = 3

ALPHA = 0.9 # Coverability
BETA = 0.1 # Confidence
GAMMA = 0.5  # Submodular decay factor for Coverability
TOTAL_STEPS = 3
# DISTANCE_THRESHOLD = 1.0
TOP_PERCENTILE_FILTER = 0.2 # Keep the top n% of viewpoints based on Score

batch_results = []

# Load Inference CSV globally to save time
if USE_GROUND_TRUTH:
    df_inference = pd.read_csv(GT_CSV_PATH)
    df_inference.rename(columns={'filename': 'Filename', 'chamfer_value': 'Predicted_CD'}, inplace=True)
else:
    df_inference = pd.read_csv(CSV_PATH)
df_inference[['Parsed_Folder', 'Viewpoint', 'Feature']] = df_inference['Filename'].str.extract(r'(.*?)/viewpoint_simulated_(\d+)_(.*?)\.pcd')
df_inference['Viewpoint'] = df_inference['Viewpoint'].astype(int)

print(f"Starting batch optimization for {len(BATCH_WORKPIECES)} workpieces...")
print(f"Method: {OPTIMIZATION_METHOD} | Steps: {TOTAL_STEPS} | Alpha: {ALPHA} | Beta: {BETA} | GT: {USE_GROUND_TRUTH}\n")

for wp in BATCH_WORKPIECES:
    print(f"{'='*50}")
    print(f"Processing Workpiece: {wp}")
    print(f"{'='*50}")
    start_time = time.time()
    
    # --- 1. SET PATHS ---
    CAD_MODEL_DIR = wp
    GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/pcd_all.pcd"
    GLOBAL_COVERED_JSON = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/covered_indices.json"
    POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}"
    
    # --- 2. LOAD DATA ---
    pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH) if os.path.exists(GLOBAL_PCD_PATH) else o3d.geometry.PointCloud()
    if os.path.exists(GLOBAL_COVERED_JSON):
        with open(GLOBAL_COVERED_JSON, "r") as f:
            global_visibility_dict = json.load(f)
    else:
        global_visibility_dict = {}
        
    poses_dict = load_viewpoint_poses_dict(POSES_PATH)
    
    df_chamfer = df_inference[df_inference['Parsed_Folder'] == wp].copy()
    if df_chamfer.empty:
        print(f"Skipping {wp} - No data in CSV!")
        continue
        
    df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)
    
    # --- NEW: Calculate Global Normalized Feature Confidence for this workpiece ---
    min_cd = df_chamfer[df_chamfer['Chamfer_Distance_mm'] >= 0]['Chamfer_Distance_mm'].min()
    max_cd = df_chamfer['Chamfer_Distance_mm'].max()
    if max_cd > min_cd:
        df_chamfer['Feature_Confidence'] = 1.0 - ((df_chamfer['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
    else:
        df_chamfer['Feature_Confidence'] = 1.0
        
    features = df_chamfer['Feature'].unique()
    
    feature_indices = {}
    all_features_indices = set()
    
    # 1. Load all features simultaneously
    feature_pcds = []
    feature_names = []
    for feat in features:
        feat_path = f"workpiece/{CAD_MODEL_DIR}/{feat}.stl"
        if os.path.exists(feat_path):
            f_mesh = o3d.io.read_triangle_mesh(feat_path)
            f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
            feature_pcds.append(f_pcd)
            feature_names.append(feat)
    
    # 2. Voronoi Nearest-Neighbor Competition
    if len(feature_pcds) > 0 and len(pcd_all.points) > 0:
        dists = np.zeros((len(pcd_all.points), len(feature_pcds)))
        for i, f_pcd in enumerate(feature_pcds):
            dists[:, i] = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
        
        assignments = np.argmin(dists, axis=1)
        
        # 3. Populate feature indices
        for i, feat in enumerate(feature_names):
            idx_set = set(np.where(assignments == i)[0])
            feature_indices[feat] = idx_set
            all_features_indices.update(idx_set)
            
    if len(all_features_indices) == 0:
        print(f"Skipping {wp} - No target feature points found!")
        continue

    # --- 3. RUN OPTIMIZATION ---
    if OPTIMIZATION_METHOD == "GREEDY":
        num_iterations = 1
        rcl_size = 1
    else:
        num_iterations = GRASP_ITERATIONS
        rcl_size = RCL_SIZE
        
    best_overall_sequence = []
    best_overall_score = -1.0
    best_point_coverage_counts = None
    
    
    # --- PRECOMPUTE STATIC CHAMFER DATA ---
    total_object_points = len(all_features_indices)
    static_viewpoint_data = {}
    temp_cameras = []
    
    for v_idx, group in df_chamfer.groupby('Viewpoint'):
        v_idx = int(v_idx)
        if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]): continue
        
        camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
        
        camera_target_indices = set()
        sum_weighted_chamfer = 0.0
        sum_points = 0
        
        for _, row in group.iterrows():
            feature = row['Feature']
            chamfer_dist = row['Chamfer_Distance_mm']
            if feature not in feature_indices: continue
                
            feat_visible_points = camera_visible_40k & feature_indices[feature]
            P_vj = len(feat_visible_points)
            if P_vj == 0: continue
                
            camera_target_indices.update(feat_visible_points)
            sum_weighted_chamfer += (P_vj * chamfer_dist)
            sum_points += P_vj
            
        if sum_points == 0: continue
            
        weighted_chamfer = sum_weighted_chamfer / sum_points
        
        temp_cameras.append({
            'v_idx': v_idx,
            'weighted_chamfer': weighted_chamfer,
            'visible_target_indices': camera_target_indices,
            'sum_points': sum_points
        })

    # Normalize and Score
    if len(temp_cameras) > 0:
        min_cam_cd = min(c['weighted_chamfer'] for c in temp_cameras)
        max_cam_cd = max(c['weighted_chamfer'] for c in temp_cameras)
        
        for c in temp_cameras:
            if max_cam_cd > min_cam_cd:
                c['confidence'] = 1.0 - ((c['weighted_chamfer'] - min_cam_cd) / (max_cam_cd - min_cam_cd))
            else:
                c['confidence'] = 1.0
                
            c['filter_score'] = (c['sum_points'] / total_object_points) * c['confidence']
            
        # Sort by Filter Score (Highest to Lowest)
        temp_cameras.sort(key=lambda x: x['filter_score'], reverse=True)
        num_to_keep = max(1, int(len(temp_cameras) * TOP_PERCENTILE_FILTER))
        top_cameras = temp_cameras[:num_to_keep]
        
        for c in top_cameras:
            static_viewpoint_data[c['v_idx']] = {
                'weighted_chamfer': c['weighted_chamfer'],
                'visible_target_indices': c['visible_target_indices'],
                'idx_arr': list(c['visible_target_indices'])
            }

    for iteration in range(1, num_iterations + 1):
        point_coverage_counts = np.zeros(len(pcd_all.points))
        selected_viewpoints = []
        cumulative_utility = 0.0
        
        for k in range(1, TOTAL_STEPS + 1):
            db_records = []
            for v_idx, data in static_viewpoint_data.items():
                if v_idx in selected_viewpoints:
                    continue
                    
                idx_arr = data['idx_arr']
                c_vals = point_coverage_counts[idx_arr]
                submodular_sum = np.sum(GAMMA ** c_vals)
                coverability = submodular_sum / len(all_features_indices)
                
                db_records.append({
                    'Viewpoint': v_idx,
                    'Coverability': coverability,
                    'Chamfer_Distance': data['weighted_chamfer'],
                    '_visible_target_indices': data['visible_target_indices']
                })
                
            df_step = pd.DataFrame(db_records)
            if df_step.empty: break
            
            min_cov = df_step['Coverability'].min()
            max_cov = df_step['Coverability'].max()
            df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
            
            df_step['Uncertainty'] = df_step['Chamfer_Distance']
            min_uncert = df_step['Uncertainty'].min()
            max_uncert = df_step['Uncertainty'].max()
            df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
            
            df_step['Information_Gain'] = df_step['Norm_Coverability']
            df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
            
            df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
            
            actual_rcl_size = min(rcl_size, len(df_step))
            rcl = df_step.head(actual_rcl_size)
            chosen_idx = random.randint(0, actual_rcl_size - 1)
            best_row = rcl.iloc[chosen_idx]
            
            cumulative_utility += best_row['Utility_Score']
            selected_viewpoints.append(int(best_row['Viewpoint']))
            
            for idx in best_row['_visible_target_indices']:
                point_coverage_counts[idx] += 1
                
        if cumulative_utility > best_overall_score:
            best_overall_score = cumulative_utility
            best_overall_sequence = selected_viewpoints
            best_point_coverage_counts = point_coverage_counts.copy()
        if OPTIMIZATION_METHOD == "GRASP":
            print(f"  Iteration {iteration}/{num_iterations} | Current Utility: {cumulative_utility:.4f} | Best Sum Utility: {best_overall_score:.4f}")
            
    process_time = time.time() - start_time
    points_covered = np.sum(best_point_coverage_counts > 0)
    total_points = len(all_features_indices)
    
    print(f"--> {wp} Done! | Utility: {best_overall_score:.4f} | Covered: {points_covered}/{total_points} | Time: {process_time:.1f}s")
    print(f"--> Best Sequence: {best_overall_sequence}\n")
    
    batch_results.append({
        'Workpiece': wp,
        'Best_Utility_Score': best_overall_score,
        'Points_Covered': f"{points_covered} / {total_points}",
        'Coverage_Percentage': f"{(points_covered / total_points * 100):.1f}%",
        'Optimal_Sequence': str(best_overall_sequence),
        'Processing_Time_s': process_time
    })

if batch_results:
    df_results = pd.DataFrame(batch_results)
    print("\n\n" + "="*60)
    print("BATCH OPTIMIZATION SUMMARY")
    print("="*60)
    display(df_results)


Starting batch optimization for 1 workpieces...
Method: GRASP | Steps: 3 | Alpha: 0.9 | Beta: 0.1 | GT: True

Processing Workpiece: TH0032AV
  Iteration 1/50 | Current Utility: 2.7434 | Best Sum Utility: 2.7434
  Iteration 2/50 | Current Utility: 2.7309 | Best Sum Utility: 2.7434
  Iteration 3/50 | Current Utility: 2.7113 | Best Sum Utility: 2.7434
  Iteration 4/50 | Current Utility: 2.7244 | Best Sum Utility: 2.7434
  Iteration 5/50 | Current Utility: 2.7244 | Best Sum Utility: 2.7434
  Iteration 6/50 | Current Utility: 2.7244 | Best Sum Utility: 2.7434
  Iteration 7/50 | Current Utility: 2.7834 | Best Sum Utility: 2.7834
  Iteration 8/50 | Current Utility: 2.7198 | Best Sum Utility: 2.7834
  Iteration 9/50 | Current Utility: 2.7786 | Best Sum Utility: 2.7834
  Iteration 10/50 | Current Utility: 2.7198 | Best Sum Utility: 2.7834
  Iteration 11/50 | Current Utility: 2.7496 | Best Sum Utility: 2.7834
  Iteration 12/50 | Current Utility: 2.7834 | Best Sum Utility: 2.7834
  Iteration 13/5

,Workpiece,Best_Utility_Score,Points_Covered,Coverage_Percentage,Optimal_Sequence,Processing_Time_s
0,TH0032AV,2.783357,32364 / 40000,80.9%,"[18, 43, 53]",24.952312


### 11. Reverse Engineering: Evaluate Manual Sequence
Use this block to manually input a sequence of cameras (like your simple geometrical grid) and see exactly how it performs in terms of raw coverage, confidence, and Chamfer distance. This is great for benchmarking the optimizer against human intuition.

In [25]:
# ==========================================
# 11. REVERSE ENGINEERING: EVALUATE MANUAL SEQUENCE
# ==========================================
# Input your manual sequence and the specific workpiece here!
EVAL_WORKPIECE = "TH0011AV"
MANUAL_SEQUENCE = [18, 89, 34, 74, 63, 135, 126, 69, 137, 65, 123, 98]

print(f"\nLoading data for {EVAL_WORKPIECE}...")

# --- 1. LOAD DATA FOR SPECIFIC WORKPIECE ---
import os
eval_pcd_path = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{EVAL_WORKPIECE}/pcd_all.pcd"
eval_cov_json = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{EVAL_WORKPIECE}/covered_indices.json"

eval_pcd = o3d.io.read_point_cloud(eval_pcd_path)
with open(eval_cov_json, "r") as f:
    eval_visibility_dict = json.load(f)

eval_df = df_inference[df_inference['Parsed_Folder'] == EVAL_WORKPIECE].copy()
if 'Predicted_CD' in eval_df.columns:
    eval_df.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)

min_cd = eval_df[eval_df['Chamfer_Distance_mm'] >= 0]['Chamfer_Distance_mm'].min()
max_cd = eval_df['Chamfer_Distance_mm'].max()
if max_cd > min_cd:
    eval_df['Feature_Confidence'] = 1.0 - ((eval_df['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
else:
    eval_df['Feature_Confidence'] = 1.0

eval_feature_indices = {} # Track individual features for point-weighting
eval_all_features_indices = set()

# 1. Load all features simultaneously
feature_pcds = []
feature_names = []
for feat in eval_df['Feature'].unique():
    feat_path = f"workpiece/{EVAL_WORKPIECE}/{feat}.stl"
    if os.path.exists(feat_path):
        f_mesh = o3d.io.read_triangle_mesh(feat_path)
        f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
        feature_pcds.append(f_pcd)
        feature_names.append(feat)

# 2. Voronoi Nearest-Neighbor Competition
if len(feature_pcds) > 0 and len(eval_pcd.points) > 0:
    dists = np.zeros((len(eval_pcd.points), len(feature_pcds)))
    for i, f_pcd in enumerate(feature_pcds):
        dists[:, i] = np.asarray(eval_pcd.compute_point_cloud_distance(f_pcd))
    
    assignments = np.argmin(dists, axis=1)
    
    # 3. Populate feature indices
    for i, feat in enumerate(feature_names):
        idx_set = set(np.where(assignments == i)[0])
        eval_feature_indices[feat] = idx_set
        eval_all_features_indices.update(idx_set)

# --- 2. EVALUATION LOOP ---
eval_coverage_counts = np.zeros(len(eval_pcd.points))
cumulative_points = 0
total_features_points = len(eval_all_features_indices)

step_breakdowns = []
total_confidence = 0.0
valid_conf_steps = 0

for step, v_idx in enumerate(MANUAL_SEQUENCE):
    camera_visible_40k = set(eval_visibility_dict.get(str(v_idx), []))
    visible_target_indices = camera_visible_40k & eval_all_features_indices
    
    new_points = 0
    for idx in visible_target_indices:
        if eval_coverage_counts[idx] == 0:
            new_points += 1
        eval_coverage_counts[idx] += 1
        
    cumulative_points += new_points
    
    cam_data = eval_df[eval_df['Viewpoint'] == v_idx]
    if cam_data.empty:
        conf = 0.0
        raw_error = 0.0
    else:
        # Point-Weighted Average (matches Block 4)
        sum_weighted_chamfer = 0.0
        sum_points = 0
        for _, row in cam_data.iterrows():
            feature = row['Feature']
            chamfer = row['Chamfer_Distance_mm']
            if feature not in eval_feature_indices: continue
            
            P_vj = len(camera_visible_40k & eval_feature_indices[feature])
            if P_vj == 0: continue
                
            sum_weighted_chamfer += (P_vj * chamfer)
            sum_points += P_vj
            
        raw_error = (sum_weighted_chamfer / sum_points) if sum_points > 0 else 0.0
        
        # Recompute normalized confidence using the weighted raw error
        if max_cd > min_cd:
            conf = 1.0 - ((raw_error - min_cd) / (max_cd - min_cd))
        else:
            conf = 1.0
            
        total_confidence += conf
        valid_conf_steps += 1
        
    cum_cov = cumulative_points / total_features_points
    util = (ALPHA * cum_cov) + (BETA * conf)
    
    step_breakdowns.append({
        'Step': step + 1,
        'Viewpoint': v_idx,
        'Utility': round(util, 4),
        'Cumulative Coverability': round(cum_cov, 4),
        'Raw Error (mm)': round(raw_error, 4),
        'Normalized Confidence': round(conf, 4)
    })

avg_sequence_conf = total_confidence / valid_conf_steps if valid_conf_steps > 0 else 0.0

print(f"\nEVALUATION COMPLETE (MANUAL SEQUENCE)\n")
print("--- PARAMETERS USED ---")
print(f"Workpiece Evaluated: {EVAL_WORKPIECE}")
print(f"Total Steps: {len(MANUAL_SEQUENCE)}")
print("-----------------------\n")
print(f"Total points covered at least once: {cumulative_points} / {total_features_points} ({(cumulative_points/total_features_points)*100:.1f}%)")
print(f"Average Confidence of Sequence: {avg_sequence_conf:.4f}")
print(f"Manual Viewpoint Sequence: {MANUAL_SEQUENCE}\n")

print("Breakdown of the Manual Sequence:")
from IPython.display import display
import pandas as pd
display(pd.DataFrame(step_breakdowns).set_index('Step'))



Loading data for TH0011AV...

EVALUATION COMPLETE (MANUAL SEQUENCE)

--- PARAMETERS USED ---
Workpiece Evaluated: TH0011AV
Total Steps: 12
-----------------------

Total points covered at least once: 32440 / 40000 (81.1%)
Average Confidence of Sequence: 0.7939
Manual Viewpoint Sequence: [18, 89, 34, 74, 63, 135, 126, 69, 137, 65, 123, 98]

Breakdown of the Manual Sequence:


,Viewpoint,Utility,Cumulative Coverability,Raw Error (mm),Normalized Confidence
Step,,,,,
1,18,0.5549,0.5303,6.2645,0.7761
2,89,0.7962,0.7941,5.5475,0.8153
3,34,0.7934,0.7947,6.1570,0.7820
4,74,0.7976,0.7967,5.7246,0.8056
5,63,0.7976,0.7984,5.9953,0.7908
6,135,0.8014,0.8033,6.1102,0.7846
7,126,0.8049,0.8061,5.9446,0.7936
8,69,0.8045,0.8069,6.1442,0.7827
9,137,0.8095,0.8084,5.4688,0.8196


In [35]:
GRASP_ITERATIONS = 50


In [36]:
# ==========================================
# 12. COMPREHENSIVE BATCH PROCESSING (ALL VIEW COUNTS)
# ==========================================
import time
import copy
import pandas as pd
import numpy as np
import random
import open3d as o3d
import os

TARGET_VIEW_COUNTS = [3, 4, 8, 12]

comprehensive_batch_results = []

print(f"Starting comprehensive batch optimization across {len(TARGET_VIEW_COUNTS)} view count settings for {len(BATCH_WORKPIECES)} workpieces...")

for target_steps in TARGET_VIEW_COUNTS:
    print(f"\n\n{'#'*60}")
    print(f"OPTIMIZING FOR TOTAL_STEPS = {target_steps}")
    print(f"{'#'*60}")
    
    for wp in BATCH_WORKPIECES:
        start_time = time.time()
        
        # --- 1. LOAD WORKPIECE DATA ---
        # Reuse the global df_inference loaded in previous cells (supports Ground Truth toggle)
        df_chamfer = df_inference[df_inference['Parsed_Folder'] == wp].copy()
        
        if df_chamfer.empty:
            print(f"Skipping {wp} - No inference data found!")
            continue
            
        pcd_all_path = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/pcd_all.pcd"
        if not os.path.exists(pcd_all_path):
            print(f"Skipping {wp} - pcd_all.pcd not found!")
            continue
        pcd_all = o3d.io.read_point_cloud(pcd_all_path)
        
        cov_json_path = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/covered_indices.json"
        with open(cov_json_path, "r") as f:
            global_visibility_dict = json.load(f)
            
        if 'Predicted_CD' in df_chamfer.columns:
            df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)
            
        min_cd = df_chamfer[df_chamfer['Chamfer_Distance_mm'] >= 0]['Chamfer_Distance_mm'].min()
        max_cd = df_chamfer['Chamfer_Distance_mm'].max()
        if max_cd > min_cd:
            df_chamfer['Feature_Confidence'] = 1.0 - ((df_chamfer['Chamfer_Distance_mm'] - min_cd) / (max_cd - min_cd))
        else:
            df_chamfer['Feature_Confidence'] = 1.0
            
        features = df_chamfer['Feature'].unique()
        
        # --- 2. VORONOI NEAREST NEIGHBOR ---
        feature_indices = {}
        all_features_indices = set()
        feature_pcds = []
        feature_names = []
        for feat in features:
            feat_path = f"workpiece/{wp}/{feat}.stl"
            if os.path.exists(feat_path):
                f_mesh = o3d.io.read_triangle_mesh(feat_path)
                f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
                feature_pcds.append(f_pcd)
                feature_names.append(feat)
                
        if len(feature_pcds) > 0 and len(pcd_all.points) > 0:
            dists = np.zeros((len(pcd_all.points), len(feature_pcds)))
            for i, f_pcd in enumerate(feature_pcds):
                dists[:, i] = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
            assignments = np.argmin(dists, axis=1)
            for i, feat in enumerate(feature_names):
                idx_set = set(np.where(assignments == i)[0])
                feature_indices[feat] = idx_set
                all_features_indices.update(idx_set)
                
        if len(all_features_indices) == 0:
            print(f"Skipping {wp} - No target feature points found!")
            continue

        # --- 3. PRECOMPUTE STATIC CHAMFER DATA ---
        total_object_points = len(all_features_indices)
        static_viewpoint_data = {}
        temp_cameras = []
        
        for v_idx, group in df_chamfer.groupby('Viewpoint'):
            v_idx = int(v_idx)
            if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]): continue
            camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
            
            camera_target_indices = set()
            sum_weighted_chamfer = 0.0
            sum_points = 0
            
            for _, row in group.iterrows():
                feature = row['Feature']
                chamfer_dist = row['Chamfer_Distance_mm']
                if feature not in feature_indices: continue
                feat_visible_points = camera_visible_40k & feature_indices[feature]
                P_vj = len(feat_visible_points)
                if P_vj == 0: continue
                camera_target_indices.update(feat_visible_points)
                sum_weighted_chamfer += (P_vj * chamfer_dist)
                sum_points += P_vj
                
            if sum_points == 0: continue
            weighted_chamfer = sum_weighted_chamfer / sum_points
            
            temp_cameras.append({
                'v_idx': v_idx,
                'weighted_chamfer': weighted_chamfer,
                'visible_target_indices': camera_target_indices,
                'sum_points': sum_points
            })

        if len(temp_cameras) > 0:
            min_cam_cd = min(c['weighted_chamfer'] for c in temp_cameras)
            max_cam_cd = max(c['weighted_chamfer'] for c in temp_cameras)
            for c in temp_cameras:
                if max_cam_cd > min_cam_cd:
                    c['confidence'] = 1.0 - ((c['weighted_chamfer'] - min_cam_cd) / (max_cam_cd - min_cam_cd))
                else:
                    c['confidence'] = 1.0
                c['filter_score'] = (c['sum_points'] / total_object_points) * c['confidence']
            
            temp_cameras.sort(key=lambda x: x['filter_score'], reverse=True)
            num_to_keep = max(1, int(len(temp_cameras) * TOP_PERCENTILE_FILTER))
            top_cameras = temp_cameras[:num_to_keep]
            
            for c in top_cameras:
                static_viewpoint_data[c['v_idx']] = {
                    'weighted_chamfer': c['weighted_chamfer'],
                    'visible_target_indices': c['visible_target_indices'],
                    'idx_arr': list(c['visible_target_indices'])
                }

        # --- 4. RUN OPTIMIZATION ---
        if OPTIMIZATION_METHOD == "GREEDY":
            num_iterations = 1
            rcl_size = 1
        else:
            num_iterations = GRASP_ITERATIONS
            rcl_size = RCL_SIZE
            
        best_overall_sequence = []
        best_overall_score = -1.0
        best_point_coverage_counts = None
        
        for iteration in range(1, num_iterations + 1):
            point_coverage_counts = np.zeros(len(pcd_all.points))
            selected_viewpoints = []
            cumulative_utility = 0.0
            
            for k in range(1, target_steps + 1):
                db_records = []
                for v_idx, data in static_viewpoint_data.items():
                    if v_idx in selected_viewpoints: continue
                    idx_arr = data['idx_arr']
                    c_vals = point_coverage_counts[idx_arr]
                    submodular_sum = np.sum(GAMMA ** c_vals)
                    coverability = submodular_sum / len(all_features_indices)
                    db_records.append({
                        'Viewpoint': v_idx,
                        'Coverability': coverability,
                        'Chamfer_Distance': data['weighted_chamfer'],
                        '_visible_target_indices': data['visible_target_indices']
                    })
                    
                df_step = pd.DataFrame(db_records)
                if df_step.empty: break
                
                min_cov = df_step['Coverability'].min()
                max_cov = df_step['Coverability'].max()
                df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
                
                df_step['Uncertainty'] = df_step['Chamfer_Distance']
                min_uncert = df_step['Uncertainty'].min()
                max_uncert = df_step['Uncertainty'].max()
                df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
                
                df_step['Information_Gain'] = df_step['Norm_Coverability']
                df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
                df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
                
                actual_rcl_size = min(rcl_size, len(df_step))
                rcl = df_step.head(actual_rcl_size)
                chosen_idx = random.randint(0, actual_rcl_size - 1)
                best_row = rcl.iloc[chosen_idx]
                
                cumulative_utility += best_row['Utility_Score']
                selected_viewpoints.append(int(best_row['Viewpoint']))
                
                for idx in best_row['_visible_target_indices']:
                    point_coverage_counts[idx] += 1
                    
            if cumulative_utility > best_overall_score:
                best_overall_score = cumulative_utility
                best_overall_sequence = selected_viewpoints
                best_point_coverage_counts = point_coverage_counts.copy()
                
        process_time = time.time() - start_time
        points_covered = np.sum(best_point_coverage_counts > 0)
        total_points = len(all_features_indices)
        
        print(f"--> {wp} [{target_steps} Views] | Utility: {best_overall_score:.4f} | Covered: {points_covered}/{total_points} | Time: {process_time:.1f}s")
        print(f"--> Sequence: {best_overall_sequence}\n")
        
        comprehensive_batch_results.append({
            'Target_View_Count': target_steps,
            'Workpiece': wp,
            'Optimal_Sequence': str(best_overall_sequence),
            'Best_Utility_Score': best_overall_score,
            'Points_Covered': f"{points_covered} / {total_points}",
            'Coverage_Percentage': f"{(points_covered / total_points * 100):.1f}%",
            'Processing_Time_s': process_time
        })

if comprehensive_batch_results:
    df_comp_results = pd.DataFrame(comprehensive_batch_results)
    print("\n\n" + "="*80)
    print("COMPREHENSIVE BATCH OPTIMIZATION SUMMARY")
    print("="*80)
    display(df_comp_results)


Starting comprehensive batch optimization across 4 view count settings for 14 workpieces...


############################################################
OPTIMIZING FOR TOTAL_STEPS = 3
############################################################
--> TH0011AV [3 Views] | Utility: 2.7375 | Covered: 31894/40000 | Time: 23.3s
--> Sequence: [34, 89, 47]

--> TH0012AV [3 Views] | Utility: 2.7716 | Covered: 32710/40000 | Time: 23.5s
--> Sequence: [18, 27, 38]

--> TH0021AV [3 Views] | Utility: 2.7564 | Covered: 31429/40000 | Time: 22.9s
--> Sequence: [21, 30, 91]

--> TH0022AV [3 Views] | Utility: 2.7849 | Covered: 31988/40000 | Time: 12.4s
--> Sequence: [13, 32, 28]

--> TH0031AV [3 Views] | Utility: 2.7449 | Covered: 31665/40000 | Time: 12.5s
--> Sequence: [13, 104, 29]

--> TH0032AV [3 Views] | Utility: 2.7839 | Covered: 32364/40000 | Time: 23.3s
--> Sequence: [18, 43, 53]

--> TH0041AV [3 Views] | Utility: 2.7524 | Covered: 31491/40000 | Time: 22.6s
--> Sequence: [43, 50, 26]

--> TH0042

,Target_View_Count,Workpiece,Optimal_Sequence,Best_Utility_Score,Points_Covered,Coverage_Percentage,Processing_Time_s
0,3,TH0011AV,"[34, 89, 47]",2.737504,31894 / 40000,79.7%,23.289346
1,3,TH0012AV,"[18, 27, 38]",2.771642,32710 / 40000,81.8%,23.512534
2,3,TH0021AV,"[21, 30, 91]",2.756368,31429 / 40000,78.6%,22.897957
3,3,TH0022AV,"[13, 32, 28]",2.784907,31988 / 40000,80.0%,12.363354
4,3,TH0031AV,"[13, 104, 29]",2.744914,31665 / 40000,79.2%,12.476732
5,3,TH0032AV,"[18, 43, 53]",2.783907,32364 / 40000,80.9%,23.333069
6,3,TH0041AV,"[43, 50, 26]",2.752393,31491 / 40000,78.7%,22.565455
7,3,TH0042AV,"[18, 58, 174]",2.758207,31985 / 40000,80.0%,22.342992
8,3,TH0051AV,"[26, 50, 41]",2.727879,31568 / 40000,78.9%,23.209540
9,3,TH0052AV,"[22, 46, 18]",2.730105,32239 / 40000,80.6%,22.795727
